In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
df = pd.read_csv('../data/raw/deliveries.csv')
print("Data loaded!")
print(f"Shape: {df.shape}")
print(f"\nColumn types before engineering:")
print(df.dtypes)

Data loaded!
Shape: (2000, 8)

Column types before engineering:
distance_km               int64
weather_severity          int64
road_type                object
season                   object
cargo_weight_kg           int64
driver_experience_yrs     int64
departure_hour            int64
is_delayed                int64
dtype: object


In [7]:
df_processed = df.copy()
le = LabelEncoder()
df_processed['road_type'] = le.fit_transform(df['road_type'])
print("Label Encoding applied to road_type")
print(f"Classes learned: {le.classes_}")
print(f"highway = {le.transform(['highway'])}")
print(f"rural   = {le.transform(['rural'])}")
print(f"\nSample of road_type column after encoding:")
print(df_processed['road_type'].head(10).values)

Label Encoding applied to road_type
Classes learned: ['highway' 'rural']
highway = [0]
rural   = [1]

Sample of road_type column after encoding:
[0 0 0 1 1 1 1 0 1 0]


In [8]:
#one hot encoding for seasons
season_dummies = pd.get_dummies(df_processed['season'], prefix='season',dtype=int)

print("One Hot Encoding applied to season")
print(f"\nNew columns created:")
print(season_dummies.columns.tolist())
print(f"\nSample of season dummies (first 5 rows):")
print(season_dummies.head())

One Hot Encoding applied to season

New columns created:
['season_fall', 'season_spring', 'season_summer', 'season_winter']

Sample of season dummies (first 5 rows):
   season_fall  season_spring  season_summer  season_winter
0            0              0              1              0
1            0              1              0              0
2            0              1              0              0
3            0              1              0              0
4            0              0              1              0


In [9]:
# Adding new columns to dataframe and dropping original season column
df_processed = pd.concat([df_processed, season_dummies], axis=1)
df_processed.drop('season', axis=1, inplace=True, errors='ignore')

print(f"\nColumns after encoding:")
print(df_processed.columns.tolist())
print(f"\nShape after encoding: {df_processed.shape}")
print(df_processed.head())


Columns after encoding:
['distance_km', 'weather_severity', 'road_type', 'cargo_weight_kg', 'driver_experience_yrs', 'departure_hour', 'is_delayed', 'season_fall', 'season_spring', 'season_summer', 'season_winter']

Shape after encoding: (2000, 11)
   distance_km  weather_severity  road_type  cargo_weight_kg  \
0          152                 3          0            18312   
1          485                 2          0             4878   
2          320                 3          0             8029   
3          156                 2          1             3571   
4          121                 5          1            17439   

   driver_experience_yrs  departure_hour  is_delayed  season_fall  \
0                     16               6           0            0   
1                     17               9           0            0   
2                      0              11           0            0   
3                     18              17           0            0   
4                   

In [10]:
# Columns that need to be normalized
cols_to_scale = ['distance_km','cargo_weight_kg','driver_experience_yrs','departure_hour']
print("Before normalization:")
print(df_processed[cols_to_scale].describe().round(2))
scaler = MinMaxScaler()
df_processed[cols_to_scale] = scaler.fit_transform(df_processed[cols_to_scale])
print("\nAfter normalization:")
print(df_processed[cols_to_scale].describe().round(2))
print("\nAll values should now be between 0 and 1")

Before normalization:
       distance_km  cargo_weight_kg  driver_experience_yrs  departure_hour
count      2000.00          2000.00                2000.00         2000.00
mean        418.93         10141.02                  14.79           12.73
std         221.03          5621.70                   8.67            5.20
min          50.00           502.00                   0.00            4.00
25%         222.00          5041.00                   7.00            8.00
50%         415.50         10249.00                  15.00           13.00
75%         611.00         14914.00                  22.00           17.00
max         799.00         19994.00                  29.00           21.00

After normalization:
       distance_km  cargo_weight_kg  driver_experience_yrs  departure_hour
count      2000.00          2000.00                2000.00         2000.00
mean          0.49             0.49                   0.51            0.51
std           0.30             0.29                   0.

In [11]:
#generating Nan values intentionally and then filling them with median 
# Step 1 — Artificially introduce 2% missing values to practice
df_with_missing = df_processed.copy()
np.random.seed(42)
for col in ['distance_km', 'cargo_weight_kg']:
    missing_indices = np.random.choice(df_with_missing.index,size=int(0.02 * len(df_with_missing)),replace=False)
    df_with_missing.loc[missing_indices, col] = np.nan

print("Missing values introduced:")
print(df_with_missing.isnull().sum())

# Step 2 — Fill missing values using median
imputer = SimpleImputer(strategy='median')
df_with_missing[cols_to_scale] = imputer.fit_transform(df_with_missing[cols_to_scale])

print("\nMissing values after imputation:")
print(df_with_missing.isnull().sum())
print("\nAll zeros — imputation successful!")

Missing values introduced:
distance_km              40
weather_severity          0
road_type                 0
cargo_weight_kg          40
driver_experience_yrs     0
departure_hour            0
is_delayed                0
season_fall               0
season_spring             0
season_summer             0
season_winter             0
dtype: int64

Missing values after imputation:
distance_km              0
weather_severity         0
road_type                0
cargo_weight_kg          0
driver_experience_yrs    0
departure_hour           0
is_delayed               0
season_fall              0
season_spring            0
season_summer            0
season_winter            0
dtype: int64

All zeros — imputation successful!
